# Q2: How can the model be configured to handle real-time requests?
**Topic:** System-design | **Difficulty:** Medium | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
How can the model be configured to handle real-time requests?

## Detailed Answer

### Definition of Real-Time
Real-time ML serving means responding to requests within a tight latency budget (typically <100ms P99).

### Architecture for Real-Time ML Serving

```
Client → Load Balancer → API Gateway → Model Server → Response
                                         ├── Model Cache (GPU memory)
                                         ├── Feature Store (Redis)
                                         └── Batching Queue
```

### Key Strategies:

#### 1. Model Optimization
| Technique | Latency Reduction | Quality Impact |
|-----------|-------------------|----------------|
| **Quantization** (INT8/FP16) | 2-4x speedup | Minimal (<1% accuracy loss) |
| **Pruning** | 2-3x speedup | Minor with fine-tuning |
| **Knowledge Distillation** | Smaller model | Depends on teacher-student gap |
| **ONNX Runtime / TensorRT** | 2-5x speedup | None (same model) |
| **Model Compilation** (torch.compile) | 1.5-3x speedup | None |

#### 2. Serving Infrastructure
- **Model Server**: Triton Inference Server, TorchServe, TF Serving
- **Dynamic Batching**: Collect multiple requests and process as a batch (better GPU utilization)
  - Wait up to X ms to fill a batch, then process — trades individual latency for throughput
- **GPU Sharing**: Multiple model replicas on one GPU with CUDA MPS
- **Model Caching**: Keep hot models in GPU memory, cold models on disk

#### 3. Feature Engineering for Speed
- **Pre-computed features**: Store in Redis/DynamoDB via feature store (Feast, Tecton)
- **Feature caching**: Cache frequently-requested user features
- **Light preprocessing**: Move heavy feature transforms offline (batch pipeline)

#### 4. Autoscaling
- **Horizontal**: Scale number of replicas based on QPS / latency
- **Vertical**: Scale GPU type based on model requirements
- **Pod autoscaler** (Kubernetes HPA) based on GPU utilization or queue depth

#### 5. Async Processing for Non-Critical Paths
- Return immediate response for core prediction
- Offload logging, A/B test recording, analytics to message queue (Kafka)

#### 6. Caching Predictions
- Cache predictions for repeated inputs (e.g., trending item recommendations)
- Use TTL-based cache invalidation
- Locality-sensitive hashing for approximate cache hits

### Monitoring for Real-Time Systems
| Metric | Target | Tool |
|--------|--------|------|
| P50 latency | <30ms | Prometheus/Grafana |
| P99 latency | <100ms | Datadog |
| Throughput (QPS) | Based on SLA | Custom |
| GPU utilization | >70% | nvidia-smi |
| Error rate | <0.1% | AlertManager |
| Model staleness | Based on domain | Feature store |

### Example: Real-Time Recommendation System
```
User Request → Feature Store (Redis, <1ms)
             → User embedding lookup (<1ms)
             → ANN search for candidates (FAISS, <5ms)
             → Ranking model inference (GPU, <20ms)
             → Business rules filter (<1ms)
             → Response (~30ms total)
```

## Key Takeaways
- **Model optimization** (quantization, TensorRT) gives the biggest single latency improvement
- **Dynamic batching** is the key to high throughput on GPUs — always use it in production
- **Feature stores** eliminate real-time feature computation — pre-compute everything possible
- Monitor **P99 latency** not average — outliers matter most for user experience